Create an employee table containing ID, name, department, salary and city.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")

print("Table created successfully!")

Table created successfully!


Insert five employee records into the employee table.

In [ ]:
cursor.executemany("""
INSERT INTO employees VALUES (?, ?, ?, ?)
""", [
    (1, "Amit", "IT", 60000),
    (2, "Priya", "HR", 50000),
    (3, "Rahul", "IT", 75000),
    (4, "Neha", "Sales", 45000),
    (5, "Riya", "HR", 55000)
])

conn.commit()
print("Five records inserted")

Five records inserted


Display all employee records.

In [ ]:
cursor.execute("SELECT * FROM employees")
print(cursor.fetchall())

[(1, 'Amit', 'IT', 60000), (2, 'Priya', 'HR', 50000), (3, 'Rahul', 'IT', 75000), (4, 'Neha', 'Sales', 45000), (5, 'Riya', 'HR', 55000)]


Find employees whose salary is greater than 50,000.

In [ ]:
cursor.execute("""
SELECT name, salary
FROM employees
WHERE salary > 50000
""")
print(cursor.fetchall())

[('Amit', 60000), ('Rahul', 75000), ('Riya', 55000)]


Display all unique departments.

In [ ]:
cursor.execute("SELECT DISTINCT department FROM employees")
print(cursor.fetchall())

[('IT',), ('HR',), ('Sales',)]


Display the two employees with the highest salaries.

In [ ]:
cursor.execute("""
SELECT name, salary
FROM employees
ORDER BY salary DESC
LIMIT 2
""")
print(cursor.fetchall())

[('Rahul', 75000), ('Amit', 60000)]


Find employees who work in IT and earn more than 60,000, or employees who work in HR.

In [ ]:
cursor.execute("""
SELECT name, department, salary
FROM employees
WHERE (department = 'IT' AND salary > 60000)
   OR department = 'HR'
""")
print(cursor.fetchall())

[('Priya', 'HR', 50000), ('Rahul', 'IT', 75000), ('Riya', 'HR', 55000)]


Find employees whose names start with R.

In [ ]:
cursor.execute("""
SELECT name FROM employees
WHERE name LIKE 'R%'
""")
print(cursor.fetchall())

[('Rahul',), ('Riya',)]


Calculate the total, average, highest, and lowest salary.

In [ ]:
cursor.execute("""
SELECT SUM(salary), AVG(salary),
       MAX(salary), MIN(salary)
FROM employees
""")
print(cursor.fetchall())

[(285000, 57000.0, 75000, 45000)]


Calculate the average salary of each department.

In [ ]:
cursor.execute("""
SELECT department, AVG(salary)
FROM employees
GROUP BY department
ORDER BY department
""")
print(cursor.fetchall())

[('HR', 52500.0), ('IT', 67500.0), ('Sales', 45000.0)]


Display departments whose average salary exceeds 50,000.

In [ ]:
cursor.execute("""
SELECT department, AVG(salary)
FROM employees
GROUP BY department
HAVING AVG(salary) > 50000
ORDER BY department
""")
print(cursor.fetchall())

[('HR', 52500.0), ('IT', 67500.0)]


Update Rahul's salary to 80,000.

In [ ]:
cursor.execute("""
UPDATE employees
SET salary = 80000
WHERE name = 'Rahul'
""")
conn.commit()

print("Rows updated:", cursor.rowcount)

Rows updated: 1


Delete the employee whose ID is 4.

In [ ]:
cursor.execute("DELETE FROM employees WHERE id = 4")
conn.commit()

print("Rows deleted:", cursor.rowcount)

Rows deleted: 1


Create a department table and display employees with their department managers.

In [ ]:
cursor.execute("""
CREATE TABLE departments (
    department TEXT PRIMARY KEY,
    manager TEXT
)
""")

cursor.executemany("""
INSERT INTO departments VALUES (?, ?)
""", [
    ("IT", "Karan"),
    ("HR", "Meera"),
    ("Sales", "Anjali")
])

cursor.execute("""
SELECT e.name, e.department, d.manager
FROM employees e
INNER JOIN departments d
ON e.department = d.department
ORDER BY e.id
""")
print(cursor.fetchall())

[('Amit', 'IT', 'Karan'), ('Priya', 'HR', 'Meera'), ('Rahul', 'IT', 'Karan'), ('Riya', 'HR', 'Meera')]


Display all employees and their department managers, including employees without a matching department.

In [ ]:
cursor.execute("""
SELECT e.name, e.department, d.manager
FROM employees e
LEFT JOIN departments d
ON e.department = d.department
ORDER BY e.id
""")
print(cursor.fetchall())

[('Amit', 'IT', 'Karan'), ('Priya', 'HR', 'Meera'), ('Rahul', 'IT', 'Karan'), ('Riya', 'HR', 'Meera')]


Find employees whose salary is greater than the current average salary.

In [ ]:
cursor.execute("""
SELECT name, salary
FROM employees
WHERE salary > (
    SELECT AVG(salary) FROM employees
)
ORDER BY salary DESC
""")
print(cursor.fetchall())

[('Rahul', 80000)]


Classify employees earning at least 60,000 as High Salary and others as Low Salary.

In [ ]:
cursor.execute("""
SELECT name, salary,
CASE
    WHEN salary >= 60000 THEN 'High Salary'
    ELSE 'Low Salary'
END AS category
FROM employees
ORDER BY id
""")
print(cursor.fetchall())

[('Amit', 60000, 'High Salary'), ('Priya', 50000, 'Low Salary'), ('Rahul', 80000, 'High Salary'), ('Riya', 55000, 'Low Salary')]


Combine the names of IT and HR employees without duplicates.

In [ ]:
cursor.execute("""
SELECT name FROM employees WHERE department = 'IT'
UNION
SELECT name FROM employees WHERE department = 'HR'
ORDER BY name
""")
print(cursor.fetchall())

[('Amit',), ('Priya',), ('Rahul',), ('Riya',)]


Use a CTE to find employees earning more than 50,000.

In [ ]:
cursor.execute("""
WITH high_salary AS (
    SELECT name, salary
    FROM employees
    WHERE salary > 50000
)
SELECT * FROM high_salary
ORDER BY salary DESC
""")
print(cursor.fetchall())

[('Rahul', 80000), ('Amit', 60000), ('Riya', 55000)]


Rank employees according to salary from highest to lowest.

In [ ]:
cursor.execute("""
SELECT name, salary,
RANK() OVER (ORDER BY salary DESC) AS salary_rank
FROM employees
ORDER BY salary DESC
""")
print(cursor.fetchall())

[('Rahul', 80000, 1), ('Amit', 60000, 2), ('Riya', 55000, 3), ('Priya', 50000, 4)]
